# Phase 0 / NB2 (v4) — Per-Technology Hardware & Communication Model

**Where we are:** NB1 built the two atomic noise channels (`dephasing_channel`,
`gate_infidelity_channel`). NB2 is the **spec sheet for each machine** — pure hardware
description, no noise applied, no fidelity numbers produced. NB3 (routing) and NB4 (lowering)
consume it.

**What NB2 produces**
- `TECHS` — SC, NA, TI specs (fidelities, T2, gate times, connectivity), hardcoded from the
  configs with the source field cited in-line. One place, one source of truth.
- `COMM` + `t_comm(a,b)` — the **communication model** (see v4 changes below).
- `HardwareModel` — coupling maps (capacity-dependent), SWAP cost, average degree, and
  builders for **distributed** machines (homogeneous references + heterogeneous pools).
- `SABRE_SEEDS` — the routing seed set, pinned here so NB3 imports one number.

---

## v4 changes — the communication model

`t_remote` is gone. It was a **single scalar used for two physically distinct quantities**:
the duration of a remote two-qubit gate, and the latency of a block-boundary state transfer.
It was `0.0`, so the conflation was harmless. It is no longer `0.0`.

**Two names, two meanings:**

| symbol | meaning | value |
|---|---|---|
| `t_comm(a,b)` | duration of a remote 2Q gate between modules of tech `a` and `b` | `max(t2q_a, t2q_b)` |
| `COMM["t_move_visible"]` | critical-path latency of a block-boundary state transfer | `0.0` |
| `COMM["f_comm"]` | aggregate fidelity of the whole teleported-gate primitive | `0.95` |
| `COMM["f_move"]` | aggregate fidelity of the whole state-transfer primitive | `0.99` |

**Why `t_comm > 0`.** A remote gate sits on the circuit's dependency path. Under pre-shared
entanglement it still requires local endpoint operations, and it cannot complete faster than the
slower endpoint's native two-qubit gate. Zero duration made a remote gate temporally *cheaper*
than a local one and hid its dominant cost: the decoherence it inflicts on **spectator** qubits
waiting behind it.

**Why `t_comm` is not uniform across architectures.** A uniform experiment-level `t_comm` would
force the homogeneous `2xSC` baseline to pay 2000 ns (1A) or 100000 ns (1B) per cross-module gate
when its own endpoints run at 200 ns — a 10x/500x handicap on the baseline, in the experiment
whose entire purpose is to beat that baseline. `t_comm` is **derived per technology pair.** The
substrate is held constant through `f_comm`; the endpoints are allowed to differ.

**Why `t_move_visible = 0`.** State transfer occurs at a synchronized block boundary, after the
moved qubit's last operation, and is assumed fully overlapped with the tail of the preceding
block. Note this is *not* free: NB4 charges every moving qubit dephasing at its **pre-move** T2
from its own `t_avail` up to the block makespan before it is allowed to leave (test ST7).
Movement is fidelity-only in *time*, never in *cost*.

**Why `f_comm` and `f_move` are uniform across module pairs.** Under heralded entanglement
distribution, channel loss and transduction inefficiency reduce the entanglement generation
*rate*, not the fidelity of a successfully heralded Bell pair; residual infidelity trades against
rate via purification. The heterogeneity penalty is therefore carried entirely by `t_comm`.
This is a forward-looking assumption and belongs in the limitations paragraph.

---

**Decisions locked (from discussion)**
- **Measurement dropped.** NB5 scores `state_fidelity(rho, ideal)` on the pre-measurement state,
  so readout infidelity never enters. Strip terminal measurements before lowering. Mid-circuit-
  measurement circuits (e.g. BV) are excluded.
- **SC = 2x2 ring** (a 4-cycle) at capacity 4: degree 2, diagonals forced through SWAPs.
  `kappa_avg = 2.0` -> set EFCL's kappa to 2.0 for these runs in Phase 2 (does not affect Phase 1).
- **Capacity is a scenario knob, not a hardware fact.** 1A = 4/module, 1B = 2/module.
- **SWAP = f2q^3 fidelity, 3*t2q duration, SC only.**
- **References are distributed.** "2xSC" = two SC modules paying `f_comm` **and** `t_comm` across
  the split — homogeneous *DQC*, not a monolithic machine.
- **Unlimited communication qubits.** Concurrent remote gates from the same module do not
  contend. Generous to heterogeneity; declare it.

In [1]:
import numpy as np
from dataclasses import dataclass, field
from itertools import combinations
from qiskit.transpiler import CouplingMap

# Routing seeds pinned here (consumed in NB3): best-of-N SABRE, take min-SWAP over these,
# so an unlucky seed never strawmans the SC baseline.
SABRE_SEEDS = list(range(10))   # N = 10
print("SABRE_SEEDS:", SABRE_SEEDS)

SABRE_SEEDS: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


## 1. Technology specs

Hardcoded from `cost_config_v3.json` (SC, NA) and `cost_config_tp2n_99.json` (SC, TI).
SC is byte-identical across both configs, so it is defined once. All times in **nanoseconds**.

`fm` / `tm` are intentionally omitted. Data-qubit measurement is terminal and stripped before
lowering, so readout never enters the score. Note that a gate teleportation *does* measure its
communication qubits mid-protocol; those measurements, the classical round trip, and the
conditional Pauli corrections are absorbed into the aggregate `f_comm` on the fidelity side and
**neglected on the timing side**. `t_comm = max(t2q_a, t2q_b)` is therefore an *optimistic lower
bound* on remote-gate duration. State it as such in the paper.

In [ ]:
@dataclass(frozen=True)
class TechSpec:
    name: str
    f1q: float           # gate_fidelity.f1q
    f2q: float           # gate_fidelity.f2q
    T2: float            # coherence.T2   (ns)  -- T1 -> inf (pure dephasing)
    t1q: float           # gate_time.t1q  (ns)
    t2q: float           # gate_time.t2q  (ns)
    kappa: float         # routing.kappa  (coarse connectivity descriptor in EFCL)
    all_to_all: bool     # routing.all_to_all
    max_qubits: int = 20 # capacity.max_qubits  (device ceiling, NOT experiment capacity)

# --- values copied from config; field cited after each ---
TECHS = {
    "sc": TechSpec(name="sc",
                   f1q=0.9999, f2q=0.999,           # v3/tp2n gate_fidelity
                   T2=80_000.0,                     # coherence.T2
                   t1q=20.0, t2q=200.0,             # gate_time
                   kappa=2.3, all_to_all=False),    # routing.kappa
    "na": TechSpec(name="na",
                   f1q=0.9995, f2q=0.997,           # v3 gate_fidelity
                   T2=200_000.0,                    # coherence.T2
                   t1q=200.0, t2q=2_000.0,          # gate_time
                   kappa=0.0, all_to_all=True),     # routing.all_to_all
    "ti": TechSpec(name="ti",
                   f1q=0.9999, f2q=0.9997,          # tp2n gate_fidelity
                   T2=2_000_000.0,                  # coherence.T2
                   t1q=10_000.0, t2q=100_000.0,     # gate_time
                   kappa=0.0, all_to_all=True),     # routing.all_to_all
}

# ---------------------------------------------------------------------------
# Communication model (v4).  ONE source of truth, consumed by NB4.
#
#   t_comm(a,b)        -- duration of a REMOTE 2Q GATE.        On the dependency path.
#   COMM["t_move_visible"] -- latency of a STATE TRANSFER.     Overlapped => 0.
#
# These are different physical events at different points in execution. They were one scalar
# (`t_remote`) until v4, which was harmless only because it was 0. Never merge them again.
# ---------------------------------------------------------------------------
COMM = {
    "f_comm":         0.95,   # aggregate fidelity of the whole teleported-gate primitive,
                              # INCLUDING its local endpoint operations. Not an add-on channel:
                              # a remote gate pays f_comm and NOT f2q.
    "f_move":         0.99,   # aggregate fidelity of the whole state-transfer primitive.
                              # Movement happens at a scheduled boundary and may consume a
                              # pre-purified Bell pair; remote gates fire on demand and consume
                              # raw pairs. Hence f_move > f_comm.
    "t_move_visible": 0.0,    # fully overlapped with the tail of the preceding block.
}

def t_comm(tech_a, tech_b):
    """Duration of a remote 2Q gate between modules of technology `tech_a` and `tech_b`.

    Gate teleportation (Eisert et al.) applies a local CNOT against a Bell-pair half at each
    endpoint; the two run in parallel. The operation cannot complete before the slower endpoint's
    native 2Q gate. Optimistic lower bound: neglects the endpoint basis measurements, the
    classical round trip, and the conditional Pauli corrections.

    Keyed on TECHNOLOGY of the two endpoints. NB4 decides *whether* a gate is remote by MODULE.
    """
    return max(TECHS[tech_a].t2q, TECHS[tech_b].t2q)

# Pin the derived values so they are version-controlled, not recomputed silently.
assert t_comm("sc","sc") ==     200.0
assert t_comm("na","na") ==   2_000.0
assert t_comm("sc","na") ==   2_000.0   # == NA local: a remote gate is time-neutral vs slow endpoint
assert t_comm("ti","ti") == 100_000.0
assert t_comm("sc","ti") == 100_000.0
assert t_comm("sc","ti") == t_comm("ti","sc")
assert COMM["t_move_visible"] == 0.0

hdr = f"{'tech':>5} {'f1q':>8} {'f2q':>8} {'T2(ns)':>10} {'t1q':>8} {'t2q(ns)':>9} {'conn':>10}"
print(hdr); print("-"*len(hdr))
for t in TECHS.values():
    conn = "all-to-all" if t.all_to_all else f"kappa={t.kappa}"
    print(f"{t.name:>5} {t.f1q:>8.4f} {t.f2q:>8.4f} {t.T2:>10.0f} {t.t1q:>8.0f} {t.t2q:>9.0f} {conn:>10}")
print(f"\ncomm: {COMM}")

### 1b. The cost of one remote gate — the table the paper is about

`f_comm` is a fixed fidelity hit, identical for every architecture. The *time* cost is not: a
remote gate runs at the **slower** endpoint's clock while the **faster** endpoint's neighbours
decohere at the **faster** endpoint's T2. Homogeneous machines never see this, because clock and
coherence are matched by construction. Heterogeneous machines see it by definition.

Read the last column. One row is not like the others.

In [ ]:
print(f"{'pair':>8} {'t_comm(ns)':>11} {'spectator T2':>13} {'t/T2':>8} {'survival':>9} "
      f"{'-log f_comm':>12} {'t/T2 nats':>10}")
print("-"*80)
_c_comm = -np.log(COMM["f_comm"])
for a, b in [("sc","sc"), ("na","na"), ("sc","na"), ("ti","ti"), ("sc","ti")]:
    tc  = t_comm(a, b)
    T2s = min(TECHS[a].T2, TECHS[b].T2)      # the spectator that suffers most
    print(f"{a+'+'+b:>8} {tc:>11.0f} {T2s:>13.0f} {tc/T2s:>8.4f} {np.exp(-tc/T2s):>9.4f} "
          f"{_c_comm:>12.4f} {tc/T2s:>10.4f}")

print("\nSC+TI: a single remote gate costs one idle spectator 1.25 nats of dephasing versus")
print(f"       {_c_comm:.4f} nats of communication infidelity -- a factor of {1.25/_c_comm:.0f}.")
print("       With t_remote = 0, EFCL saw only the 0.0513. That is what this change fixes.")

## 2. Connectivity — coupling map as a function of capacity

SC is a **cycle** on its qubits: cap 4 → the 2×2 ring (0-1-2-3-0), cap 2 → a single edge
(no routing), cap 1 → an isolated node. NA/TI return `None` = all-to-all (NB3 reads `None`
as "no routing, no SWAPs"). The `avg_degree` helper is what you set EFCL's κ to for
consistency in Phase 2.

In [3]:
class HardwareModel:
    def __init__(self, techs, comm):
        self.techs = techs
        self.comm = comm

    def spec(self, tech):
        return self.techs[tech]

    def coupling_map(self, tech, n_qubits):
        """Local coupling map (0..n-1). None => all-to-all (no routing)."""
        t = self.techs[tech]
        if t.all_to_all:
            return None
        if n_qubits <= 1:
            return CouplingMap([])                       # single node, no edges
        if n_qubits == 2:
            return CouplingMap([[0, 1], [1, 0]])         # single edge, no routing
        # cycle on n qubits; n_qubits == 4 is exactly the locked 2x2 ring
        edges = []
        for i in range(n_qubits):
            j = (i + 1) % n_qubits
            edges += [[i, j], [j, i]]
        return CouplingMap(edges)

    def avg_degree(self, tech, n_qubits):
        """Average undirected degree of the local topology (all-to-all -> n-1)."""
        t = self.techs[tech]
        if t.all_to_all:
            return float(n_qubits - 1)
        cm = self.coupling_map(tech, n_qubits)
        und = {frozenset(e) for e in cm.get_edges() if e[0] != e[1]}
        return 2.0 * len(und) / n_qubits if n_qubits else 0.0

    # ---- SWAP cost (SC only; all-to-all techs never SWAP) ----
    def swap_fidelity(self, tech):
        """SC has no native SWAP: 3 CX -> f2q**3."""
        return self.techs[tech].f2q ** 3

    def swap_duration(self, tech):
        return 3.0 * self.techs[tech].t2q

HW = HardwareModel(TECHS, COMM)

print("SC coupling maps by capacity:")
for cap in (1, 2, 4):
    cm = HW.coupling_map("sc", cap)
    edges = sorted({tuple(sorted(e)) for e in cm.get_edges()}) if cm else None
    print(f"  cap={cap}: edges={edges}  avg_degree={HW.avg_degree('sc', cap):.2f}")
print(f"NA cap=4 coupling_map: {HW.coupling_map('na', 4)}  (None = all-to-all, "
      f"avg_degree={HW.avg_degree('na', 4):.1f})")
print(f"\nSC SWAP: fidelity=f2q^3={HW.swap_fidelity('sc'):.6f}  "
      f"duration=3*t2q={HW.swap_duration('sc'):.0f} ns")

SC coupling maps by capacity:
  cap=1: edges=[]  avg_degree=0.00
  cap=2: edges=[(0, 1)]  avg_degree=1.00
  cap=4: edges=[(0, 1), (0, 3), (1, 2), (2, 3)]  avg_degree=2.00
NA cap=4 coupling_map: None  (None = all-to-all, avg_degree=3.0)

SC SWAP: fidelity=f2q^3=0.997003  duration=3*t2q=600 ns


## 3. Distributed machines (references + heterogeneous pools)

A `Machine` is a list of `Module`s with **global** qubit indexing; cross-module 2Q
interactions are remote (charged `f_comm` in NB4 — there is no coupling edge between modules).
`homogeneous_machine("sc", 2, 4)` is the **2×SC** reference (8 qubits, distributed DQC);
`heterogeneous_machine([("sc",4),("na",4)])` is the mixed pool for 1A.

In [4]:
@dataclass
class Module:
    module_id: int
    tech: str
    qubits: tuple          # global qubit indices owned by this module
    coupling_map: object   # local CouplingMap or None (all-to-all)

    @property
    def n_qubits(self):
        return len(self.qubits)

@dataclass
class Machine:
    name: str
    modules: list

    @property
    def n_qubits(self):
        return sum(m.n_qubits for m in self.modules)

    def module_of(self, global_qubit):
        for m in self.modules:
            if global_qubit in m.qubits:
                return m
        raise KeyError(global_qubit)

    def summary(self):
        parts = [f"{m.tech}[{m.qubits[0]}..{m.qubits[-1]}]" for m in self.modules]
        return f"{self.name}: {self.n_qubits}q = " + " | ".join(parts)

def _build(name, layout):
    """layout: list of (tech, cap). Assigns contiguous global indices per module."""
    modules, base = [], 0
    for mid, (tech, cap) in enumerate(layout):
        qubits = tuple(range(base, base + cap))
        modules.append(Module(mid, tech, qubits, HW.coupling_map(tech, cap)))
        base += cap
    return Machine(name, modules)

def homogeneous_machine(tech, n_modules, cap_per_module):
    return _build(f"{n_modules}x{tech.upper()}", [(tech, cap_per_module)] * n_modules)

def heterogeneous_machine(layout):
    tag = "+".join(f"{t.upper()}" for t, _ in layout)
    return _build(tag, layout)

# ---- 1A scenarios: equal capacity, 2 modules x cap 4 = 8 qubits ----
print("1A (SC+NA, cap 4/module, 8q):")
for M in [homogeneous_machine("sc", 2, 4),
          homogeneous_machine("na", 2, 4),
          heterogeneous_machine([("sc", 4), ("na", 4)])]:
    print("  " + M.summary())

# ---- 1B scenarios: 2 modules x cap 2 = 4 qubits (SC module = single edge, no routing) ----
print("\n1B (SC+TI, cap 2/module, 4q):")
for M in [homogeneous_machine("sc", 2, 2),
          homogeneous_machine("ti", 2, 2),
          heterogeneous_machine([("sc", 2), ("ti", 2)])]:
    print("  " + M.summary())

1A (SC+NA, cap 4/module, 8q):
  2xSC: 8q = sc[0..3] | sc[4..7]
  2xNA: 8q = na[0..3] | na[4..7]
  SC+NA: 8q = sc[0..3] | na[4..7]

1B (SC+TI, cap 2/module, 4q):
  2xSC: 4q = sc[0..1] | sc[2..3]
  2xTI: 4q = ti[0..1] | ti[2..3]
  SC+TI: 4q = sc[0..1] | ti[2..3]


## 4. Physics landscape (self-documentation)

Echoes the ratios that decide the experiments, straight from the specs — the notebook records
*why* each pair is used where.

In [5]:
sc, na, ti = TECHS["sc"], TECHS["na"], TECHS["ti"]
print("Idle penalty an SC qubit pays waiting through one foreign 2Q gate (t_gate / T2_SC):")
print(f"  through NA 2Q ({na.t2q:.0f} ns): {na.t2q/sc.T2:.4f} of T2  -> survives exp(-x)={np.exp(-na.t2q/sc.T2):.3f}")
print(f"  through TI 2Q ({ti.t2q:.0f} ns): {ti.t2q/sc.T2:.4f} of T2  -> survives exp(-x)={np.exp(-ti.t2q/sc.T2):.3f}")
print("\nParking a qubit on the low-decoherence tech (idle survival per unit idle):")
print(f"  1A uses SC+NA: NA T2 {na.T2:.0f} vs SC T2 {sc.T2:.0f} -> {na.T2/sc.T2:.1f}x  (moderate)")
print(f"  1B uses SC+TI: TI T2 {ti.T2:.0f} vs SC T2 {sc.T2:.0f} -> {ti.T2/sc.T2:.1f}x  (decisive)")
print("\nRaw 2Q gate fidelity: SC {:.4f}, NA {:.4f}, TI {:.4f}".format(sc.f2q, na.f2q, ti.f2q))
print("=> TI dominates SC on fidelity+coherence (no static spatial trade-off) => SC+TI is 1B, not 1A.")
print("=> SC vs NA trade off (SC faster/higher-f, NA all-to-all + longer T2) => SC+NA is 1A.")

Idle penalty an SC qubit pays waiting through one foreign 2Q gate (t_gate / T2_SC):
  through NA 2Q (2000 ns): 0.0250 of T2  -> survives exp(-x)=0.975
  through TI 2Q (100000 ns): 1.2500 of T2  -> survives exp(-x)=0.287

Parking a qubit on the low-decoherence tech (idle survival per unit idle):
  1A uses SC+NA: NA T2 200000 vs SC T2 80000 -> 2.5x  (moderate)
  1B uses SC+TI: TI T2 2000000 vs SC T2 80000 -> 25.0x  (decisive)

Raw 2Q gate fidelity: SC 0.9990, NA 0.9970, TI 0.9997
=> TI dominates SC on fidelity+coherence (no static spatial trade-off) => SC+TI is 1B, not 1A.
=> SC vs NA trade off (SC faster/higher-f, NA all-to-all + longer T2) => SC+NA is 1A.


## 5. Optional config drift-check (skippable)

Hardcoding is fine for speed, but this cell re-reads the configs *if present* and asserts the
hardcoded values still match — so the notebook can't silently desync from the source of truth.
Skips cleanly when the config files aren't on this machine.

In [ ]:
import json, os

def _drift_check():
    candidates = ["/mnt/project", ".", "..", "./configs"]
    def find(fn):
        for d in candidates:
            p = os.path.join(d, fn)
            if os.path.exists(p): return p
        return None

    checked = 0
    for fn, names in [("cost_config_v3.json", ["sc", "na"]),
                      ("cost_config_tp2n_99.json", ["sc", "ti"])]:
        p = find(fn)
        if p is None:
            print(f"  {fn}: not found, skipping"); continue
        cfg = json.load(open(p))
        by = {t["name"].lower(): t for t in cfg["techs"]}
        for nm in names:
            c, t = by[nm], TECHS[nm]
            assert np.isclose(c["gate_fidelity"]["f1q"], t.f1q), f"{nm} f1q drift"
            assert np.isclose(c["gate_fidelity"]["f2q"], t.f2q), f"{nm} f2q drift"
            assert np.isclose(c["coherence"]["T2"], t.T2), f"{nm} T2 drift"
            assert np.isclose(c["gate_time"]["t2q"], t.t2q), f"{nm} t2q drift"
            checked += 1
        assert np.isclose(cfg["comm"]["f_comm"], COMM["f_comm"])
        assert np.isclose(cfg["comm"]["f_move"], COMM["f_move"])
        # DELIBERATE DIVERGENCE, not drift: EFCL still runs t_remote = 0. The Aer harness now
        # uses a nonzero, per-pair t_comm. Adding t_comm to EFCL requires block-ASAP (a global
        # per-layer clock would charge every idle qubit the full 100 us) and therefore a full
        # retrain. That decision is gated on the Phase-2 divergence measurement. Assert the
        # config is still 0 so this divergence stays intentional and visible.
        assert np.isclose(cfg["comm"].get("t_remote", 0.0), 0.0), (
            "cost_config t_remote is no longer 0 -- EFCL and the Aer harness have both changed. "
            "Reconcile deliberately: t_comm in EFCL is unsafe without block-ASAP.")
        print(f"  {fn}: OK ({', '.join(names)})  [t_remote=0 in EFCL: intentional divergence]")
    print(f"drift-check passed ({checked} tech-configs verified)" if checked
          else "drift-check skipped (no configs on this machine)")

_drift_check()

## 6. Checkpoint — NB2 go/no-go

Asserts the hardcoded values, the capacity-dependent SC topology, SWAP cost, and distributed
reference layout. NB3 imports `HW`, `TECHS`, `SABRE_SEEDS`.

In [ ]:
def _checkpoint():
    # hardcode spot-checks
    assert TECHS["sc"].f2q == 0.999 and TECHS["sc"].T2 == 80_000.0
    assert TECHS["na"].t2q == 2_000.0 and TECHS["ti"].t2q == 100_000.0
    assert TECHS["ti"].T2 == 2_000_000.0
    # SC 2x2 ring at cap 4: 4 undirected edges, avg degree 2.0
    ring = {tuple(sorted(e)) for e in HW.coupling_map("sc", 4).get_edges()}
    assert ring == {(0, 1), (1, 2), (2, 3), (0, 3)}, ring
    assert np.isclose(HW.avg_degree("sc", 4), 2.0)
    assert np.isclose(HW.avg_degree("sc", 2), 1.0)
    assert HW.coupling_map("na", 4) is None and HW.coupling_map("ti", 2) is None
    assert np.isclose(HW.swap_fidelity("sc"), 0.999**3)
    assert np.isclose(HW.swap_duration("sc"), 600.0)
    M = homogeneous_machine("sc", 2, 4)
    assert M.n_qubits == 8 and [m.qubits for m in M.modules] == [(0,1,2,3), (4,5,6,7)]
    assert [m.tech for m in M.modules] == ["sc", "sc"]
    H_ = heterogeneous_machine([("sc", 4), ("na", 4)])
    assert [m.tech for m in H_.modules] == ["sc", "na"]

    # ---- v4: communication model ----
    assert "t_remote" not in COMM, "t_remote must not exist: it conflated two quantities"
    assert set(COMM) == {"f_comm", "f_move", "t_move_visible"}
    assert COMM["t_move_visible"] == 0.0
    assert COMM["f_move"] > COMM["f_comm"]         # scheduled boundary => pre-purified Bell pair
    assert t_comm("sc","sc") == 200.0              # homogeneous SC gets a CHEAP remote gate
    assert t_comm("sc","ti") == 100_000.0
    assert t_comm("sc","ti") == t_comm("ti","sc")  # symmetric
    for a in TECHS:                                # never faster than the slower endpoint
        for b in TECHS:
            assert t_comm(a,b) >= max(TECHS[a].t2q, TECHS[b].t2q)
    # the one asymmetry that carries the paper
    assert t_comm("sc","ti") / TECHS["sc"].T2 > 1.0
    assert t_comm("sc","sc") / TECHS["sc"].T2 < 0.01
    return True

print("NB2 CHECKPOINT PASSED" if _checkpoint() else "FAILED")